In [1]:
# Basics
import numpy as np
import glob
import os
import sys

# Data Handling
import xarray as xr
import intake

# Processing
from cdo import Cdo
from tqdm import tqdm 
import logging
import re

# plotting

import matplotlib.pyplot as plt
import matplotlib.path as mpath
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Load own functions
import custom_logger_functions as lgfct
import search_cmip6_catalog as searching

cdo = Cdo()

# load catalog
catalog = intake.open_esm_datastore("/work/ik1017/Catalogs/dkrz_cmip6_disk.json") # This takes a while to load...

print("DONE")

/sw/spack-levante/mambaforge-22.9.0-2-Linux-x86_64-kptncg/lib/python3.10/site-packages/intake_esm/cat.py:264: DtypeWarning: Columns (21,22,23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


DONE


In [6]:
### Set Parameters

## set user information

group = ''
username = ''

# Select dimension of output data
dimensions = "1D" #

# Create a list of variables for processing
variables = ["sitemptop"]#["sitempsnic"]#["siitdconc"]

scenarios = ["historical", "ssp245", "ssp370"] #, "ssp126", "ssp245", "ssp370", "ssp585"]

# CREATE TERMINAL LOGGER that prints information based on the logging level
# logger will always print the statements according to the level plus all information from higher levels
#    e.g. logging level "info" will print all "info", "warning" and "error" text but not "debug"
#
# debug:   all print statement like file names from grid area, raw data ... or CDO command (great for debugging :) )
# info:    nice to know prints to see how the process is going (printing model, ensemble member, ...)
# warning: can't find files from the same ensemble member when combining variables
# error:   something is seriously wrong eg. variable name is not defined in the processing chain or a file should be there but isn't

logging_level = "debug"
logger = lgfct.build_terminal_logger(logging_level, logger_name="processing")

### Set Paths

In [7]:
outpath        = "/work/"+group+"/"+username+"/CMIP6_sea_ice/Sea_ice_temp_evolution/"                        # test outpath

gridpath       = "/work/"+group+"/DATA/modelling/CMIP6/gridareas/"

AO_mask        = "/work/"+group+"/"+username+"/CMIP6_sea_ice/Sea_ice_temp_evolution/Arctic_ocean_mask_regions.nc"                    # Arctic Ocean Mask
mask_var_names = {"ocean":"arctic_mask"} # name of the variable in the dataset containing the mask

ocean_maskpath = "/work/"+group+"/"+username+"/CMIP6_sea_ice/Sea_ice_temp_evolution/ocean-masks-remap/"                    # where the remapped to the model grid masks will be stored

# !!! The masks will be remapped to each model grid. 
#     To save computing time the code will NOT perform the remapping each time but only if the remapped file is not available
#     If you change the mask here you have to delete all remapped mask files so that they will be calculated again from the new masks

## Processing Functions

In [8]:
def file_existence(filepath: str, logger: logging.Logger):
    """
    Checks if a file exists and logs a message accordingly.

    Parameters:
    - filepath (str): The path to the file to check.
    - logger (logging.Logger): The logger object to use for logging messages.

    """

    filename = filepath.split("/")[-1]

    if os.path.isfile(filepath):
        logger.debug(f"File '{filename}' exists.")
    else:
        logger.error(f"File '{filename}' does not exist.")

In [9]:
def retrieve_file_names(model: str, modelcenter: str, ensemblemember: str, scenario: str, variable: str, table_id: str, activity_id: str):
    """
    Constructs file paths and retrieves the names of input and area file for a specified CMIP6 model, ensemble member and variable.

    Parameters:
        model (str): The name of the CMIP6 model.
        modelcenter (str): The center or institution responsible for the model.
        ensemblemember (str): The ensemble member identifier (e.g., "r1i1p1f1").
        scenario (str): The emission scenario or experiment (e.g., "ssp585").
        variable (str): The variable of interest (e.g., "fgco2").
        table_id (str): The CMIP6 table identifier (e.g., "Amon").
        activity_id (str): The activity or project ID (e.g., "CMIP").

    Returns:
        - inputfile (str or None): File path to the NetCDF input file(s) matching the provided criteria, or None if the file is missing.
        - area_file (str or None): File path to the area data file, or None if the area file is missing.

    Notes:
        - The function searches for available grid types and versions within the constructed directory.
        - If the "v1" version is available, it prioritizes it; otherwise, it selects the highest available version.
    """
    
    # Construct filepath
    inpath = f"/pool/data/CMIP6/data/{activity_id}/{modelcenter}/{model}/{scenario}/{ensemblemember}/{table_id}/{variable}/" 
    logger.debug(f"inpath:     {inpath}")

    # Check if files exist in path
    folders_in_path = [os.path.basename(x) for x in glob.glob(inpath + '/*')]
    if not folders_in_path:
        logger.error(f"Filepath doesn't exist ({inpath})")
        return None, None
    
    # Get grid type and available versions
    gridtype = folders_in_path[0]
    av_versions = [os.path.basename(x) for x in glob.glob(f"{inpath}{gridtype}/v*")]
    
    # Determine version
    if 'v1' in av_versions:
        version = 'v1'
    else:
        version = 'v'+str(np.max([int(v[1::]) for v in av_versions]))
    
    inputfile = f"{inpath}{gridtype}/{version}/*.nc"

    # Find grid area file
    try:
        area_file = glob.glob(gridpath + f"{model}_{gridarea_var}_{gridtype}*")[0]
        logger.debug("area file:  " +area_file)
    except:
        logger.error(f"area file is missing: {gridpath}{model}_{gridarea_var}_{gridtype}*")
        area_file = None
        
    logger.debug("input file: " +inputfile)

    return inputfile, area_file

## Processing Loop

In [10]:
for variable in variables[:]:
    for scenario in scenarios[:]:
        logger.info(scenario)

        # search the catalog
        activity_id, table_id, modellist, modelcenters, ensemblemembers, unit = searching.modelsearch(catalog, scenario, variable, logger, member=None)

        print(activity_id, table_id, modellist, modelcenters, ensemblemembers, unit)

historical
VARIABLE:  sitemptop
table_id:  SImon
long_name: Surface Temperature of Sea Ice
units:     {'K'}
dummy:     /work/ik1017/CMIP6/data/CMIP6/CMIP/AWI/AWI-ESM-1-1-LR/historical/r1i1p1f1/SImon/sitemptop/gn/v20200212/sitemptop_SImon_AWI-ESM-1-1-LR_historical_r1i1p1f1_gn_185001-185012.nc
mean for t=0: 268.7452697753906

ssp245


CMIP SImon ['ACCESS-CM2', 'AWI-ESM-1-1-LR', 'AWI-ESM-1-REcoM', 'BCC-CSM2-MR', 'BCC-ESM1', 'CAMS-CSM1-0', 'CAS-ESM2-0', 'CESM2', 'CESM2-FV2', 'CESM2-WACCM', 'CESM2-WACCM-FV2', 'CIESM', 'CMCC-CM2-SR5', 'CMCC-ESM2', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'CanESM5', 'CanESM5-CanOE', 'E3SM-1-0', 'E3SM-1-1', 'E3SM-1-1-ECA', 'EC-Earth3', 'EC-Earth3-AerChem', 'EC-Earth3-CC', 'EC-Earth3-Veg', 'EC-Earth3-Veg-LR', 'FGOALS-f3-L', 'FGOALS-g3', 'FIO-ESM-2-0', 'GFDL-CM4', 'GFDL-ESM4', 'GISS-E2-1-G', 'GISS-E2-1-H', 'HadGEM3-GC31-LL', 'HadGEM3-GC31-MM', 'IPSL-CM6A-LR', 'KIOST-ESM', 'MIROC6', 'MPI-ESM-1-2-HAM', 'MPI-ESM1-2-HR', 'MPI-ESM1-2-LR', 'MRI-ESM2-0', 'NESM3', 'NorESM2-LM', 'NorESM2-MM', 'SAM0-UNICON', 'UKESM1-0-LL'] {'ACCESS-CM2': 'CSIRO-ARCCSS', 'AWI-ESM-1-1-LR': 'AWI', 'AWI-ESM-1-REcoM': 'AWI', 'BCC-CSM2-MR': 'BCC', 'BCC-ESM1': 'BCC', 'CAMS-CSM1-0': 'CAMS', 'CAS-ESM2-0': 'CAS', 'CESM2': 'NCAR', 'CESM2-FV2': 'NCAR', 'CESM2-WACCM': 'NCAR', 'CESM2-WACCM-FV2': 'NCAR', 'CIESM': 'THU', 'CMCC-

VARIABLE:  sitemptop
table_id:  SImon
long_name: Surface Temperature of Sea Ice
units:     {'K'}
dummy:     /work/ik1017/CMIP6/data/CMIP6/ScenarioMIP/AWI/AWI-ESM-1-REcoM/ssp245/r1i1p1f1/SImon/sitemptop/gn/v20230314/sitemptop_SImon_AWI-ESM-1-REcoM_ssp245_r1i1p1f1_gn_201501-203412.nc
mean for t=0: 269.5221862792969

ssp370


ScenarioMIP SImon ['ACCESS-CM2', 'AWI-ESM-1-REcoM', 'BCC-CSM2-MR', 'CAMS-CSM1-0', 'CESM2', 'CESM2-WACCM', 'CIESM', 'CMCC-CM2-SR5', 'CMCC-ESM2', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'CanESM5', 'CanESM5-CanOE', 'EC-Earth3', 'EC-Earth3-CC', 'EC-Earth3-Veg', 'EC-Earth3-Veg-LR', 'FGOALS-f3-L', 'FGOALS-g3', 'FIO-ESM-2-0', 'GFDL-CM4', 'GFDL-ESM4', 'GISS-E2-1-G', 'HadGEM3-GC31-LL', 'IPSL-CM6A-LR', 'KACE-1-0-G', 'KIOST-ESM', 'MIROC6', 'MPI-ESM1-2-HR', 'MPI-ESM1-2-LR', 'MRI-ESM2-0', 'NESM3', 'NorESM2-LM', 'NorESM2-MM', 'UKESM1-0-LL'] {'ACCESS-CM2': 'CSIRO-ARCCSS', 'AWI-ESM-1-REcoM': 'AWI', 'BCC-CSM2-MR': 'BCC', 'CAMS-CSM1-0': 'CAMS', 'CESM2': 'NCAR', 'CESM2-WACCM': 'NCAR', 'CIESM': 'THU', 'CMCC-CM2-SR5': 'CMCC', 'CMCC-ESM2': 'CMCC', 'CNRM-CM6-1': 'CNRM-CERFACS', 'CNRM-CM6-1-HR': 'CNRM-CERFACS', 'CNRM-ESM2-1': 'CNRM-CERFACS', 'CanESM5': 'CCCma', 'CanESM5-CanOE': 'CCCma', 'EC-Earth3': 'EC-Earth-Consortium', 'EC-Earth3-CC': 'EC-Earth-Consortium', 'EC-Earth3-Veg': 'EC-Earth-Consortium', 'EC

VARIABLE:  sitemptop
table_id:  SImon
long_name: Surface Temperature of Sea Ice
units:     {'K'}
dummy:     /work/ik1017/CMIP6/data/CMIP6/ScenarioMIP/BCC/BCC-CSM2-MR/ssp370/r1i1p1f1/SImon/sitemptop/gn/v20190315/sitemptop_SImon_BCC-CSM2-MR_ssp370_r1i1p1f1_gn_201501-210012.nc
mean for t=0: 270.9933776855469



ScenarioMIP SImon ['ACCESS-CM2', 'BCC-CSM2-MR', 'CAMS-CSM1-0', 'CESM2', 'CESM2-WACCM', 'CMCC-CM2-SR5', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CanESM5', 'CanESM5-CanOE', 'EC-Earth3', 'EC-Earth3-AerChem', 'EC-Earth3-Veg', 'EC-Earth3-Veg-LR', 'FGOALS-f3-L', 'FGOALS-g3', 'GFDL-ESM4', 'GISS-E2-1-G', 'IPSL-CM6A-LR', 'KACE-1-0-G', 'MIROC6', 'MPI-ESM-1-2-HAM', 'MPI-ESM1-2-HR', 'MPI-ESM1-2-LR', 'MRI-ESM2-0', 'NorESM2-LM', 'NorESM2-MM', 'UKESM1-0-LL'] {'ACCESS-CM2': 'CSIRO-ARCCSS', 'BCC-CSM2-MR': 'BCC', 'CAMS-CSM1-0': 'CAMS', 'CESM2': 'NCAR', 'CESM2-WACCM': 'NCAR', 'CMCC-CM2-SR5': 'CMCC', 'CNRM-CM6-1': 'CNRM-CERFACS', 'CNRM-CM6-1-HR': 'CNRM-CERFACS', 'CanESM5': 'CCCma', 'CanESM5-CanOE': 'CCCma', 'EC-Earth3': 'EC-Earth-Consortium', 'EC-Earth3-AerChem': 'EC-Earth-Consortium', 'EC-Earth3-Veg': 'EC-Earth-Consortium', 'EC-Earth3-Veg-LR': 'EC-Earth-Consortium', 'FGOALS-f3-L': 'CAS', 'FGOALS-g3': 'CAS', 'GFDL-ESM4': 'NOAA-GFDL', 'GISS-E2-1-G': 'NASA-GISS', 'IPSL-CM6A-LR': 'IPSL', 'KACE-1-0-G': 'NIMS-KMA', 'MI

In [11]:
for variable in variables[:]:
    for scenario in scenarios[:]:
        logger.info(scenario)

        # search the catalog
        activity_id, table_id, modellist, modelcenters, ensemblemembers, unit = searching.modelsearch(catalog, scenario, variable, logger, member=None)

        # set the grid area variable
        gridarea_var = "areacello" 

        for model in tqdm(modellist[:], leave=True):#["MPI-ESM1-2-LR"]
            logger.info(model)
            
            for ensemblemember in ensemblemembers[model][:1]:
                outputfile = f"{outpath}{variable}/{variable}_masked_{model}_{ensemblemember}_{scenario}_{dimensions}.nc" 

                if not os.path.isdir(outputfile.rsplit('/',1)[0]):
                    os.system('mkdir '+outputfile.rsplit('/',1)[0])
                
                try:
                    logger.info("... "+ensemblemember)
                    
                    inputfile, area_file,  = retrieve_file_names(model, modelcenters[model], ensemblemember, scenario, variable, table_id, activity_id)

                    SIC_inputfile, SIC_area_file = retrieve_file_names(model, modelcenters[model], ensemblemember, scenario, "siconc", table_id, activity_id)

                    
                    bottemp_flag = False
                    try:
                        bottemp_inputfile, bottemp_area_file = retrieve_file_names(model, modelcenters[model], ensemblemember, scenario, "sitempbot", table_id, activity_id)
                        if bottemp_inputfile:
                            bottemp_flag = True
                        else:
                            bottemp_flag = False
                    except:
                        bottemp_flag = False
                    
                    if area_file != SIC_area_file:
                        logger.debug(f"{area_file} vs {SIC_area_file}")
                        logger.error(f"something wrong with the SIC area file")
                    
                    ## mask

                    if gridarea_var == "areacello": # ocean grid
                        remapped_mask = ocean_maskpath + f"{model}_arctic-ocean-mask_remapped.nc"
                        mask_var      = mask_var_names["ocean"]
                        mask_file     = AO_mask
                    else:
                        logger.error(f"something wrong with the grid")
                    if True:
                        # Check if remapped mask exists for selected model (if not remap original mask to model grid)
                        if not os.path.isfile(remapped_mask):
                            examplefile = glob.glob(inputfile)[0] 
                            logger.debug(f"Remapping mask using {examplefile}")
                            logger.debug(f"masking cmd:  -remapbil,{examplefile} -selvar,mask {mask_file} "+ remapped_mask)
                            cdo.copy(input=f" -remapbil,{area_file} -selvar,{mask_var} {mask_file}", output=remapped_mask)
                            file_existence(remapped_mask, logger)
    
                        logger.debug(remapped_mask)
    
                        outputfile = f"{outpath}{variable}/{variable}_AO_{model}_{ensemblemember}_{scenario}.nc"
                        SIC_outputfile = f"{outpath}siconc/siconc_AO_{model}_{ensemblemember}_{scenario}.nc"
                        area_outputfile = f"{outpath}area_crops/area_AO_{model}_{ensemblemember}_{scenario}.nc"
                        bottemp_outputfile = f"{outpath}sitempbot/sitempbot_AO_{model}_{ensemblemember}_{scenario}_1D.nc"
                        SIA_outputfile = f"{outpath}siconc/siarea_AO_{model}_{ensemblemember}_{scenario}_gt15SIC.nc"
                        outputfile_ice = f"{outpath}{variable}/{variable}_AO_{model}_{ensemblemember}_{scenario}_SICgt15.nc"
                        surftemp_outputfile = f"{outpath}{variable}/{variable}_AO_{model}_{ensemblemember}_{scenario}_surf_temp_1D.nc"
                        surftemp_weighted_outputfile = f"{outpath}{variable}/{variable}_AO_{model}_{ensemblemember}_{scenario}_surf_temp_weighted_by_SIC_1D.nc"
                        
                        logger.debug(outputfile)
    

                        # select Arctic region
                        cdo.copy(input=f"-sellonlatbox,0,360,60,90 -ifthen {remapped_mask} -mergetime {inputfile}", output=outputfile, options="--reduce_dim")
                        cdo.copy(input=f"-sellonlatbox,0,360,60,90 -ifthen {remapped_mask} -mergetime {SIC_inputfile}", output=SIC_outputfile, options="--reduce_dim")
                        cdo.copy(input=f"-sellonlatbox,0,360,60,90 -ifthen {remapped_mask} {area_file}", output=area_outputfile, options="--reduce_dim")

                        # add bottemp to output if available
                        if bottemp_flag:
                            cdo.copy(input=f"-sellonlatbox,0,360,60,90 -ifthen {remapped_mask} -mergetime {bottemp_inputfile}", output=bottemp_outputfile, options="--reduce_dim")

                        # some SIC are outputted in percent some in 1, here we derive the correct threshold and apply it to derive a masked SIC file and Sea ice area
                        if np.nanmax(cdo.fldmax(input=f"-selvar,siconc {SIC_outputfile}", returnMaArray='siconc'))>2: 
                            threshold=15
                            print(model, threshold, np.nanmax(cdo.fldmax(input=f"-selvar,siconc {SIC_outputfile}", returnMaArray='siconc')))
                        else:
                            threshold=0.15
                            print(model, threshold, np.nanmax(cdo.fldmax(input=f"-selvar,siconc {SIC_outputfile}", returnMaArray='siconc')))
                        
                        if threshold==15:
                            cdo.copy(input=f"-mul -divc,100 -ifthen -gtc,{threshold} {SIC_outputfile} {SIC_outputfile} {area_outputfile}", output=SIA_outputfile , options="--reduce_dim")
                        else:
                            cdo.copy(input=f"-mul -ifthen -gtc,{threshold} {SIC_outputfile} {SIC_outputfile} {area_outputfile}", output=SIA_outputfile , options="--reduce_dim")

                        cdo.copy(input=f"-ifthen -gtc,{threshold} {SIC_outputfile} {outputfile}", output=outputfile_ice, options="--reduce_dim")

                        # output average surface temperature and weighted average surface temperature (self-calculated)
                        cdo.copy(input=f"-setname,{variable} -fldmean {outputfile_ice}", output=surftemp_outputfile , options="--reduce_dim")
                        
                        cdo.copy(input=f"-setname,{variable} -div -fldsum -mul {SIA_outputfile} {outputfile_ice} -fldsum {SIA_outputfile}", output=surftemp_weighted_outputfile , options="--reduce_dim")

                        # clean up a bit
                        os.system(f"rm {outputfile_ice}")
                                                
                except:
                    print('error with model: ', model)
                    #continue

historical
VARIABLE:  sitemptop
table_id:  SImon
long_name: Surface Temperature of Sea Ice
units:     {'K'}
dummy:     /work/ik1017/CMIP6/data/CMIP6/CMIP/AWI/AWI-ESM-1-1-LR/historical/r1i1p1f1/SImon/sitemptop/gn/v20200212/sitemptop_SImon_AWI-ESM-1-1-LR_historical_r1i1p1f1_gn_185001-185012.nc
mean for t=0: 268.7452697753906

  0%|          | 0/48 [00:00<?, ?it/s]ACCESS-CM2
... r1i1p1f1
inpath:     /pool/data/CMIP6/data/CMIP/CSIRO-ARCCSS/ACCESS-CM2/historical/r1i1p1f1/SImon/sitemptop/
area file:  /work/uo1227/DATA/modelling/CMIP6/gridareas/ACCESS-CM2_areacello_gn.nc
input file: /pool/data/CMIP6/data/CMIP/CSIRO-ARCCSS/ACCESS-CM2/historical/r1i1p1f1/SImon/sitemptop/gn/v20200817/*.nc
inpath:     /pool/data/CMIP6/data/CMIP/CSIRO-ARCCSS/ACCESS-CM2/historical/r1i1p1f1/SImon/siconc/
area file:  /work/uo1227/DATA/modelling/CMIP6/gridareas/ACCESS-CM2_areacello_gn.nc
input file: /pool/data/CMIP6/data/CMIP/CSIRO-ARCCSS/ACCESS-CM2/historical/r1i1p1f1/SImon/siconc/gn/v20200817/*.nc
inpath:     /pool/

ACCESS-CM2 15 99.99993


  2%|▏         | 1/48 [00:16<12:54, 16.48s/it]AWI-ESM-1-1-LR
... r1i1p1f1
inpath:     /pool/data/CMIP6/data/CMIP/AWI/AWI-ESM-1-1-LR/historical/r1i1p1f1/SImon/sitemptop/
area file:  /work/uo1227/DATA/modelling/CMIP6/gridareas/AWI-ESM-1-1-LR_areacello_gn_calc.nc
input file: /pool/data/CMIP6/data/CMIP/AWI/AWI-ESM-1-1-LR/historical/r1i1p1f1/SImon/sitemptop/gn/v20200212/*.nc
inpath:     /pool/data/CMIP6/data/CMIP/AWI/AWI-ESM-1-1-LR/historical/r1i1p1f1/SImon/siconc/
area file:  /work/uo1227/DATA/modelling/CMIP6/gridareas/AWI-ESM-1-1-LR_areacello_gn_calc.nc
input file: /pool/data/CMIP6/data/CMIP/AWI/AWI-ESM-1-1-LR/historical/r1i1p1f1/SImon/siconc/gn/v20200212/*.nc
inpath:     /pool/data/CMIP6/data/CMIP/AWI/AWI-ESM-1-1-LR/historical/r1i1p1f1/SImon/sitempbot/
Filepath doesn't exist (/pool/data/CMIP6/data/CMIP/AWI/AWI-ESM-1-1-LR/historical/r1i1p1f1/SImon/sitempbot/)
/work/uo1227/u241314/CMIP6_sea_ice/Sea_ice_temp_evolution/ocean-masks-remap/AWI-ESM-1-1-LR_arctic-ocean-mask_remapped.nc
/work/uo12

AWI-ESM-1-1-LR 15 100.0


  4%|▍         | 2/48 [00:36<14:22, 18.75s/it]AWI-ESM-1-REcoM
... r1i1p1f1
inpath:     /pool/data/CMIP6/data/CMIP/AWI/AWI-ESM-1-REcoM/historical/r1i1p1f1/SImon/sitemptop/
area file is missing: /work/uo1227/DATA/modelling/CMIP6/gridareas/AWI-ESM-1-REcoM_areacello_gn*
input file: /pool/data/CMIP6/data/CMIP/AWI/AWI-ESM-1-REcoM/historical/r1i1p1f1/SImon/sitemptop/gn/v20230314/*.nc
inpath:     /pool/data/CMIP6/data/CMIP/AWI/AWI-ESM-1-REcoM/historical/r1i1p1f1/SImon/siconc/
Filepath doesn't exist (/pool/data/CMIP6/data/CMIP/AWI/AWI-ESM-1-REcoM/historical/r1i1p1f1/SImon/siconc/)
inpath:     /pool/data/CMIP6/data/CMIP/AWI/AWI-ESM-1-REcoM/historical/r1i1p1f1/SImon/sitempbot/
Filepath doesn't exist (/pool/data/CMIP6/data/CMIP/AWI/AWI-ESM-1-REcoM/historical/r1i1p1f1/SImon/sitempbot/)
Remapping mask using /pool/data/CMIP6/data/CMIP/AWI/AWI-ESM-1-REcoM/historical/r1i1p1f1/SImon/sitemptop/gn/v20230314/sitemptop_SImon_AWI-ESM-1-REcoM_historical_r1i1p1f1_gn_199001-200912.nc
masking cmd:  -remapbil,/po

...

NorESM2-LM 15 99.99992


 93%|█████████▎| 26/28 [07:47<00:19,  9.63s/it]NorESM2-MM
... r1i1p1f1
inpath:     /pool/data/CMIP6/data/ScenarioMIP/NCC/NorESM2-MM/ssp370/r1i1p1f1/SImon/sitemptop/
area file:  /work/uo1227/DATA/modelling/CMIP6/gridareas/NorESM2-MM_areacello_gn_calc.nc
input file: /pool/data/CMIP6/data/ScenarioMIP/NCC/NorESM2-MM/ssp370/r1i1p1f1/SImon/sitemptop/gn/v20191108/*.nc
inpath:     /pool/data/CMIP6/data/ScenarioMIP/NCC/NorESM2-MM/ssp370/r1i1p1f1/SImon/siconc/
area file:  /work/uo1227/DATA/modelling/CMIP6/gridareas/NorESM2-MM_areacello_gn_calc.nc
input file: /pool/data/CMIP6/data/ScenarioMIP/NCC/NorESM2-MM/ssp370/r1i1p1f1/SImon/siconc/gn/v20191108/*.nc
inpath:     /pool/data/CMIP6/data/ScenarioMIP/NCC/NorESM2-MM/ssp370/r1i1p1f1/SImon/sitempbot/
area file:  /work/uo1227/DATA/modelling/CMIP6/gridareas/NorESM2-MM_areacello_gn_calc.nc
input file: /pool/data/CMIP6/data/ScenarioMIP/NCC/NorESM2-MM/ssp370/r1i1p1f1/SImon/sitempbot/gn/v20191108/*.nc
/work/uo1227/u241314/CMIP6_sea_ice/Sea_ice_temp_evolutio

NorESM2-MM 15 99.99993


 96%|█████████▋| 27/28 [08:02<00:11, 11.14s/it]UKESM1-0-LL
... r1i1p1f2
inpath:     /pool/data/CMIP6/data/ScenarioMIP/MOHC/UKESM1-0-LL/ssp370/r1i1p1f2/SImon/sitemptop/
area file is missing: /work/uo1227/DATA/modelling/CMIP6/gridareas/UKESM1-0-LL_areacello_gr*
input file: /pool/data/CMIP6/data/ScenarioMIP/MOHC/UKESM1-0-LL/ssp370/r1i1p1f2/SImon/sitemptop/gr/v20190510/*.nc
inpath:     /pool/data/CMIP6/data/ScenarioMIP/MOHC/UKESM1-0-LL/ssp370/r1i1p1f2/SImon/siconc/
area file:  /work/uo1227/DATA/modelling/CMIP6/gridareas/UKESM1-0-LL_areacello_gn_calc.nc
input file: /pool/data/CMIP6/data/ScenarioMIP/MOHC/UKESM1-0-LL/ssp370/r1i1p1f2/SImon/siconc/gn/v20200319/*.nc
inpath:     /pool/data/CMIP6/data/ScenarioMIP/MOHC/UKESM1-0-LL/ssp370/r1i1p1f2/SImon/sitempbot/
area file:  /work/uo1227/DATA/modelling/CMIP6/gridareas/UKESM1-0-LL_areacello_gn_calc.nc
input file: /pool/data/CMIP6/data/ScenarioMIP/MOHC/UKESM1-0-LL/ssp370/r1i1p1f2/SImon/sitempbot/gn/v20200319/*.nc
None vs /work/uo1227/DATA/modelling/C

Error in calling operator copy with:
>>> /sw/spack-levante/mambaforge-22.9.0-2-Linux-x86_64-kptncg/bin/cdo -O -s --reduce_dim -copy -sellonlatbox,0,360,60,90 -ifthen /work/uo1227/u241314/CMIP6_sea_ice/Sea_ice_temp_evolution/ocean-masks-remap/UKESM1-0-LL_arctic-ocean-mask_remapped.nc None /work/uo1227/u241314/CMIP6_sea_ice/Sea_ice_temp_evolution/area_crops/area_AO_UKESM1-0-LL_r1i1p1f2_ssp370.nc<<<
STDOUT:
STDERR:cdo(2) ifthen: Open failed on >None<
               No such file or directory

error with model:  UKESM1-0-LL
